# 大师投研室数理机制说明

本文档面向开发者解释项目从输入股票到输出组合的主要数学和规则机制。系统不是投资建议工具，而是一个多 Agent 投研原型。

## 1. 总体链路

`输入股票和资金 -> 数据层 -> 信号层 -> 因子层 -> 大师 Agent -> MAAD 辩论 -> 投资经理裁判 -> 组合执行`

核心思想是让 LLM 参与判断和表达，但关键数值边界由可复现规则控制。

## 2. 输入与交易约束

用户输入最多 6 支股票，资金池范围为：

$$10000 \le Capital \le 10000000$$

A 股交易采用一手约束：

$$LotSize = 100$$

最终买入股数为 100 的整数倍。若预算不足一手，则实际买入 0 股。

## 3. 通用因子向量

系统将技术、财务、新闻信号归一化为大师都可见的因子：`value`、`quality`、`growth`、`risk`、`technical`、`momentum`、`sentiment`、`event_heat`、`cashflow`、`volatility_control`、`sentiment_contrarian` 等。

估值因子：

$$Value = 1 - clip(0.55 \times PE/65 + 0.45 \times PB/9, 0, 1)$$

质量因子：

$$Quality = clip(0.50 \times ROE/30 + 0.34 \times CashflowCoverage/2.6 + 0.16 \times FundamentalScore, 0, 1)$$

成长因子：

$$Growth = clip((GrowthRaw + 10)/52, 0, 1)$$

风险因子：

$$Risk = clip(1 - I(RSI>75)0.24 - max(BollingerPosition-0.82,0)0.7 - max(0.45-FundamentalScore,0), 0, 1)$$

## 4. 大师规则基线

每位大师先计算规则基线分数。该分数是 LLM 决策的锚点，也是 LLM 不可用时的 fallback。

- Buffett：强调质量、现金流、估值和长期风险。
- Graham：强调低估值、安全边际和财务底线。
- Lynch：强调 GARP，即合理估值下的成长。
- Soros：强调技术趋势、情绪、事件热度和反身性。
- Dalio：强调风险均衡、质量、估值和波动控制。
- Templeton：强调逆向价值、风险补偿和情绪修复。

## 5. LLM 决策与规则护栏

真实 LLM 启用后，大师输出严格 JSON：`action`、`score`、`confidence`、`reason`。系统会将 LLM 分数限制在规则基线附近：

$$Score_{guarded} = clip(Score_{llm}, BaselineScore-\Delta, BaselineScore+\Delta)$$

默认：

$$\Delta = 0.18$$

动作阈值：

$$Score \ge 0.64 \Rightarrow Buy$$

$$Score \le 0.41 \Rightarrow Sell$$

其余为 Hold。LLM 失败或 JSON 不合法时回退规则结果。

## 6. MAAD 参与者选择

每只股票最多 3 位大师参与辩论。触发强度：

$$Trigger = 0.45D_{style} + 0.40|Score_i-\bar{Score}| + 0.10Confidence_i + ActionPressure$$

风格距离采用余弦距离：

$$D_{style}(a,b)=1-\frac{a\cdot b}{||a||\,||b||}$$

候选条件包括：风格距离超过阈值、动作不同于多数派、或评分明显偏离均值。

## 7. MAAD 攻防修正

攻击强度：

$$Attack = clip(0.28 + 0.26D_{style} + 0.34ScoreGap + ActionGap + TranscriptPressure, 0.08, 0.95)$$

防守强度：

$$Defense = clip(0.26 + 0.32MeanPreferredFactors + 0.22|Score-0.5| + 0.22ConfidenceEdge - Fatigue, 0.08, 0.95)$$

信心变化：

$$\Delta Confidence = clip((Defense-Attack)0.13 - Penalty, -0.085, 0.075)$$

分数变化来自 `Defense - Attack`，只作用于该大师最强的两个偏好因子，并被限制在小范围内。

## 8. 投资经理裁判

裁判使用辩论后的分数和信心度。

信心加权 alpha：

$$Alpha = \frac{\sum_i clip(Confidence_i,0.15,0.98)Score_i}{\sum_i clip(Confidence_i,0.15,0.98)}$$

分歧度：

$$Disagreement = std(Score_1,...,Score_6)$$

数据质量：

$$DataQuality = 0.40PriceQuality + 0.32FundamentalQuality + 0.28NewsQuality$$

风险惩罚：

$$RiskPenalty = clip(0.16VolatilityScaled + 0.14Disagreement + 0.18(1-DataQuality),0,0.5)$$

最终分：

$$FinalScore = max(Alpha - RiskPenalty - 0.32,0)$$

## 9. 组合权重

只有 `FinalScore > 0` 且最终动作为 `buy` 的股票进入可投资集合。权重按最终分比例分配，并受单股上限约束：

$$Weight_i \propto FinalScore_i$$

$$Weight_i \le 0.60$$

最终股数：

$$Shares_i = \left\lfloor \frac{Budget_i}{Price_i \times 100} \right\rfloor \times 100$$

剩余部分保留为现金。